# exp_2 on Kaggle: B_strat seeds 1 and 2

Two more seeds of the best model (stratified sampler + CE), so B_strat's 89.7 balanced accuracy can be reported as a
mean over 3 seeds. The configs are B_strat_s0 with only `save_name` changed; the seed comes from `OPENGAIT_SEED`.

**Settings before running**
- Accelerator: **GPU T4 x2** (one seed per GPU, ~7.5 h; with one GPU the runs go one after the other and do not fit in 12 h)
- Input: the `scoliosis-1k` dataset (folder `Scoliosis1K-sil-pkl`, 1493 sequences)
- Internet: **on**
- Run with **Save Version -> Save & Run All (Commit)** so it keeps going with the browser closed

Output: `/kaggle/working/exp2_kaggle_Bseeds.tgz` -> put it in `checkpoint_2/experiments/exp_2/runs/incoming/kaggle/` on the laptop.

## Step 1. Get the code

Clones branch `exp2-kaggle`. If the repo is private, add a Kaggle secret `GH_TOKEN` (GitHub fine-grained token,
Contents: read, this repo only); it is used only if it exists and is never printed.

In [ ]:
import os, subprocess
REPO_URL = 'github.com/dindawijayanto/scoliosis-gait-reproduction.git'
BRANCH = 'exp2-kaggle'
DEST = '/kaggle/working/OpenGait'

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    pass
url = f'https://{token}@{REPO_URL}' if token else f'https://{REPO_URL}'
if not os.path.isdir(DEST):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, DEST], check=True)
print(subprocess.run(['git', '-C', DEST, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## Step 2. Check the GPUs and the data

In [ ]:
!nvidia-smi -L
import glob
pkl = glob.glob('/kaggle/input/**/Scoliosis1K-sil-pkl/**/*.pkl', recursive=True)
print('pkl sequences:', len(pkl), '(expected 1493)')

## Step 3. Train B_strat_s1 and B_strat_s2, export the logits, pack the results

Runs `checkpoint_2/experiments/exp_2/code/kaggle_Bseeds.sh`. Progress lines appear here; the full training logs are in
`checkpoint_2/experiments/exp_2/runs/logs/`.

In [ ]:
!bash /kaggle/working/OpenGait/checkpoint_2/experiments/exp_2/code/kaggle_Bseeds.sh

## Step 4. Check the result

Each run should have val and test logits at 12k / 14k / 16k plus the mirror and deskew probes at 16k.

In [ ]:
!ls -la /kaggle/working/exp2_kaggle_Bseeds.tgz
!find /kaggle/working/OpenGait/checkpoint_2/experiments/exp_2/runs/logits -name '*.npz' | sort
!tail -2 /kaggle/working/OpenGait/checkpoint_2/experiments/exp_2/runs/logs/B_strat_s1.log /kaggle/working/OpenGait/checkpoint_2/experiments/exp_2/runs/logs/B_strat_s2.log